## **Tool Calling Using Ollama**

In [24]:
import ollama

In [25]:
inventory_db = {
    "laptop": {"stock": 5, "base_price": 1200},
    "monitor": {"stock": 0, "base_price": 300},
    "keyboard": {"stock": 25, "base_price": 80}
}

In [26]:
## Defining tool
## Tool 1: Checking Database

def check_inventory(product_name):
    product_name = product_name.lower()

    if product_name in inventory_db:
        return inventory_db[product_name]

    return {"stock": 0, "base_price": None}


## Tool 2: Business logic for discounts

def calculate_loyalty_discount(base_price, years_as_customer):
    discount = min(years_as_customer * 0.05, 0.30)
    final_price = base_price * (1 - discount)

    return round(final_price, 2)

In [27]:
## Map the function so that the script can call them by None

available_functions = {
    "check_inventory": check_inventory,
    "calculate_loyalty_discount": calculate_loyalty_discount
}

In [28]:
## Defining Schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Get stock and price for a product",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string"}
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_loyalty_discount",
            "description": "Calculate final price based on loyalty years",
            "parameters": {
                "type": "object",
                "properties": {
                    "base_price": {"type": "number"},
                    "years_as_customer": {"type": "integer"}
                },
                "required": ["base_price", "years_as_customer"]
            }
        }
    }
]

In [29]:
tools

[{'type': 'function',
  'function': {'name': 'check_inventory',
   'description': 'Get stock and price for a product',
   'parameters': {'type': 'object',
    'properties': {'product_name': {'type': 'string'}},
    'required': ['product_name']}}},
 {'type': 'function',
  'function': {'name': 'calculate_loyalty_discount',
   'description': 'Calculate final price based on loyalty years',
   'parameters': {'type': 'object',
    'properties': {'base_price': {'type': 'number'},
     'years_as_customer': {'type': 'integer'}},
    'required': ['base_price', 'years_as_customer']}}}]

In [30]:
## I am a customer for 5 years, what will be the final price of the laptop
## I want to buy an iphone, can you check stock

message = [
    {
        "role": "user",
        "content": "I am a customer for 5 years. What will be the final price of a laptop ?"
    }
]


## Calling LLM

response = ollama.chat(
    model="llama3.2:latest",
    messages=message,
    tools=tools
)

print(response)

model='llama3.2:latest' created_at='2026-11-17T06:51:04.1336243Z' done=True done_reason='stop' total_duration=2109932400 load_duration=37179100 prompt_eval_count=230 prompt_eval_duration=110196000 eval_count=25 eval_duration=1949395000 message=Message(role='assistant', content='{"name":"calculate_loyalty_discount","parameters{"base_price":800, "years_as_customer":5}"}}', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None


In [31]:
print(response["message"])

role='assistant' content='{"name":"calculate_loyalty_discount","parameters{"base_price":800, "years_as_customer":5}"}}' thinking=None images=None tool_name=None tool_calls=None


In [32]:
## Check if the model asked to use any tools
tool_calls = response['message'].get('tool_calls')

if tool_calls:
    for tool_call in tool_calls:
        tool_name = tool_call['function']['name']
        tool_args = tool_call['function']['arguments']

        # Get the actual Python function
        function_to_call = available_functions[tool_name]

        # Run the function with given arguments
        result = function_to_call(**tool_args)

        # Add the model's tool request to the conversation
        message.append(response['message'])

        # Add the tool's result back to the conversation
        message.append({
            'role': 'tool',
            'content': str(result)
        })

message

[{'role': 'user',
  'content': 'I am a customer for 5 years. What will be the final price of a laptop ?'}]

In [33]:
## Ask the model to generate the final answer
final_response = ollama.chat(
    model='llama3.2:latest',
    messages=message
)

print(final_response['message']['content'])

I can't provide a specific final price for a laptop without knowing the current market prices and the discounts being offered. I can give you a rough estimate of how discounts are typically calculated based on the length of time you've been a customer.

For a loyal customer like you, the discount could be anywhere from 5% to 20% off the original price, depending on the retailer and their loyalty program. Here's a rough estimate of how the discount could work:

* 5% discount: 5% of the original price = $50-$100 off
* 10% discount: 10% of the original price = $100-$200 off
* 15% discount: 15% of the original price = $150-$300 off
* 20% discount: 20% of the original price = $200-$400 off

Assuming an original price of $1,000, the discounted price could be:

* $950-$1,000 (5% discount)
* $900-$1,000 (10% discount)
* $850-$1,000 (15% discount)
* $800-$1,000 (20% discount)

Please note that these are rough estimates and the actual discount you'll receive may vary depending on the retailer an